# Q-Learning Strategy Analysis

This notebook analyses the optional tabular Q-learning strategy for the Cop agent.

## Bellman update

The Q-value for state $s$ and action $a$ is updated with:

$$ Q(s,a) \leftarrow Q(s,a) + \alpha \big[ r + \gamma \max_{a'} Q(s',a') - Q(s,a) \big] $$

where $\alpha$ is the learning rate, $\gamma$ the discount factor, $r$ the immediate reward,
and $s'$ the next state.

## References
- Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press.
- Watkins, C. J. C. H. (1989). Learning from delayed rewards. PhD thesis, University of Cambridge.


In [1]:
import matplotlib.pyplot as plt
import numpy as np
from copthief.agents.training import train_cop_q_learning

GRID = (5, 5)
THIEF_POS = (4, 4)
EPISODES = 500
WINDOW = 20


## Learning curve

In [2]:
strategy, history = train_cop_q_learning(
    GRID, THIEF_POS, episodes=EPISODES, seed=0
)

def moving_average(values, window=WINDOW):
    return np.convolve(values, np.ones(window)/window, mode='valid')

plt.figure(figsize=(8, 4))
plt.plot(history, alpha=0.3, label='raw reward')
plt.plot(range(WINDOW-1, EPISODES), moving_average(history), label=f'{WINDOW}-episode moving average')
plt.xlabel('Episode')
plt.ylabel('Total reward')
plt.title('Cop Q-learning learning curve')
plt.legend()
plt.tight_layout()
plt.show()


## Sensitivity analysis — learning rate $\alpha$

In [3]:
alphas = [0.01, 0.05, 0.1, 0.3, 0.5]
final_rewards = []
for alpha in alphas:
    _, history = train_cop_q_learning(GRID, THIEF_POS, episodes=EPISODES, alpha=alpha, seed=0)
    final_rewards.append(np.mean(history[-50:]))

plt.figure(figsize=(6, 4))
plt.plot(alphas, final_rewards, marker='o')
plt.xlabel('Learning rate $\alpha$')
plt.ylabel('Mean final reward (last 50 episodes)')
plt.title('Sensitivity to learning rate')
plt.tight_layout()
plt.show()


## Sensitivity analysis — discount factor $\gamma$

In [4]:
gammas = [0.0, 0.5, 0.8, 0.9, 0.99]
final_rewards_gamma = []
for gamma in gammas:
    _, history = train_cop_q_learning(GRID, THIEF_POS, episodes=EPISODES, gamma=gamma, seed=0)
    final_rewards_gamma.append(np.mean(history[-50:]))

plt.figure(figsize=(6, 4))
plt.plot(gammas, final_rewards_gamma, marker='s')
plt.xlabel('Discount factor $\gamma$')
plt.ylabel('Mean final reward (last 50 episodes)')
plt.title('Sensitivity to discount factor')
plt.tight_layout()
plt.show()


## Sensitivity analysis — exploration $\epsilon$

In [5]:
epsilons = [0.0, 0.05, 0.1, 0.3, 0.5]
final_rewards_epsilon = []
for epsilon in epsilons:
    _, history = train_cop_q_learning(GRID, THIEF_POS, episodes=EPISODES, epsilon=epsilon, seed=0)
    final_rewards_epsilon.append(np.mean(history[-50:]))

plt.figure(figsize=(6, 4))
plt.plot(epsilons, final_rewards_epsilon, marker='^')
plt.xlabel('Exploration rate $\epsilon$')
plt.ylabel('Mean final reward (last 50 episodes)')
plt.title('Sensitivity to exploration rate')
plt.tight_layout()
plt.show()


## Heatmap of learned state values

In [6]:
strategy, _ = train_cop_q_learning(GRID, THIEF_POS, episodes=EPISODES, seed=0)
values = np.max(strategy.q, axis=1).reshape(GRID)

plt.figure(figsize=(5, 4))
plt.imshow(values, origin='lower', cmap='viridis')
plt.colorbar(label='Max Q-value')
plt.title('Learned state-value function ($\max_a Q(s,a)$)')
plt.xlabel('Column')
plt.ylabel('Row')
plt.tight_layout()
plt.show()
